Visualisierungen der einzelnen Klassen

In [12]:
file_path = "whatamidoing.geojson"

Anzahl Geschosse

In [2]:
import geopandas as gpd
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import json

# ── LOAD DATA FROM GEOJSON ────────────────────────────────────────────────────
gdf = gpd.read_file(
    "../whatamidoing.geojson"
)

print(gdf.columns.tolist())


# ── KEY FIX: fill NaN BEFORE value_counts ────────────────────────────────────
col = "gastw"   # change if the name differs in the GeoJSON
series = gdf[col].fillna("Keine Angabe").astype(str)

counts = series.value_counts().reset_index()
counts.columns = ["Geschosse", "Anzahl"]

def sort_key(val):
    try:
        return (0, float(val))
    except:
        return (1, val)

counts = counts.sort_values("Geschosse", key=lambda x: x.map(sort_key)).reset_index(drop=True)

# ── CHART ─────────────────────────────────────────────────────────────────────
fig = go.Figure(go.Bar(
    x=counts["Geschosse"],
    y=counts["Anzahl"],
    textposition="outside",
))

fig.update_traces(cliponaxis=False)
fig.update_layout(
    title={
        "text": "Anzahl Geschosse – Verteilung CH (GeoJSON)"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                "Quelle: projected_buildings_enriched_ch.geojson | inkl. fehlende Werte</span>"
    }
)
fig.update_xaxes(title_text="Geschosse", type="category")
fig.update_yaxes(title_text="Anzahl Gebäude")

fig.write_image("anzahl_geschosse_geojson.png")
with open("anzahl_geschosse_geojson.png.meta.json", "w") as f:
    json.dump({
        "caption": "Verteilung Anzahl Geschosse (GeoJSON, inkl. fehlende Werte)",
        "description": "Bar chart of floor count distribution from projected_buildings_enriched_ch.geojson including missing values"
    }, f)

print(counts.to_string(index=False))


['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
   Geschosse  Anzahl
         1.0  314730
         2.0  868238
         3.0  589106
         4.0  184955
         5.0   71280
         6.0   30704
         7.0   15764
         8.0    7575
         9.0    3543
        10.0    1581
        11.0     755
        12.0     515
        13.0     382
        14.0     252
        15.0     200
        16.0     145
        17.0      73
        18.0 

In [11]:
buildingClass_count = gdf
print(buildingClass_count.columns.tolist())

# print first 10 entries of buildingClass_count:
print(buildingClass_count["buildingStatus"].head(10))

['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
0    1004
1    1007
2    1004
3    1004
4    1004
5    1004
6    1004
7    1004
8    1004
9    1004
Name: buildingStatus, dtype: int32


In [10]:
# Quick check for 'buildingCategory' coverage
buildingCategory_count = gdf['buildingCategory'].notna().sum()
total_count = len(gdf)
buildingCategory_percent = (buildingCategory_count / total_count) * 100 if total_count > 0 else 0

print(f"Entries with a value for 'buildingCategory': {buildingCategory_count} out of {total_count} ({buildingCategory_percent:.2f}%)")

Entries with a value for 'buildingCategory': 3348785 out of 3349331 (99.98%)


Anzahl EGIDS

In [6]:
import geopandas as gpd
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file(
    "whatamidoing.geojson"
)

print(gdf.columns.tolist())

# ── COUNT EGID COVERAGE ───────────────────────────────────────────────────────
# Normalize EGID column first (handles '', ' ', '0', etc.)
egid_raw = gdf["egid"].astype(str).str.strip()

has_egid = egid_raw.notna() & (egid_raw != "") & (egid_raw != "0")
total = len(gdf)
count_with    = has_egid.sum()
count_without = total - count_with
pct_with    = count_with / total * 100
pct_without = count_without / total * 100

print(f"Total features:    {total:,}")
print(f"Mit EGID:          {count_with:,}  ({pct_with:.1f}%)")
print(f"Ohne EGID:         {count_without:,} ({pct_without:.1f}%)")


# ── PIE CHART ─────────────────────────────────────────────────────────────────
labels = ["Mit EGID", "Ohne EGID"]
values = [count_with, count_without]

fig = go.Figure(go.Pie(
    labels=labels,
    values=values,
    textinfo="label+percent+value",
    texttemplate="%{label}<br>%{percent}<br>%{value:,}",
    hole=0.3,
))

fig.update_layout(
    title={
        "text": "EGID Abdeckung –  Gebäude CH"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                f"Quelle: AV/GWR | Total: {total:,} Gebäude</span>"
    },
    uniformtext_minsize=14,
    uniformtext_mode="hide",
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="right", x=1.1)
)

fig.write_image("egid_coverage.png")
with open("egid_coverage.png.meta.json", "w") as f:
    json.dump({
        "caption": "EGID Abdeckung Gebäude Schweiz",
        "description": "Pie chart showing how many buildings have a GWR_EGID value vs missing"
    }, f)


['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
Total features:    3,349,331
Mit EGID:          3,349,331  (100.0%)
Ohne EGID:         0 (0.0%)


Anzahl Stockwerke

In [5]:
import geopandas as gpd
import plotly.graph_objects as go
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file(
    "whatamidoing.geojson"
)

print(gdf.columns.tolist())

# ── COUNT ─────────────────────────────────────────────────────────────────────
total = len(gdf)
has_val = gdf["gastw"].notna()
count_with    = has_val.sum()
count_without = (~has_val).sum()
pct_with    = count_with / total * 100
pct_without = count_without / total * 100

print(f"Total:          {total:,}")
print(f"Mit Wert:       {count_with:,}  ({pct_with:.1f}%)")
print(f"Keine Angabe:   {count_without:,} ({pct_without:.1f}%)")

# ── PIE CHART ─────────────────────────────────────────────────────────────────
fig = go.Figure(go.Pie(
    labels=["Mit Wert", "Keine Angabe"],
    values=[count_with, count_without],
    textinfo="label+percent",
    texttemplate="%{label}<br>%{percent}<br>%{value:,}",
    hole=0.3,
))

fig.update_layout(
    title={
        "text": "Anzahl Geschosse – Datenvollständigkeit CH"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                f"Quelle: AV/GWR | Total: {total:,} Gebäude</span>"
    },
    uniformtext_minsize=14,
    uniformtext_mode="hide",
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="right", x=1.1)
)

fig.write_image("geschosse_coverage.png")
with open("geschosse_coverage.png.meta.json", "w") as f:
    json.dump({
        "caption": "Datenvollständigkeit Anzahl Geschosse",
        "description": "Pie chart showing share of buildings with and without a floor count value"
    }, f)


['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
Total:          3,349,331
Mit Wert:       2,090,125  (62.4%)
Keine Angabe:   1,259,206 (37.6%)


stockwerk value by kanton prozentual

In [4]:
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file(
    "whatamidoing.geojson"
)

print(gdf.columns.tolist())
gdf["gastw"] = pd.to_numeric(gdf["gastw"], errors="coerce")

# ── COUNT PER CANTON ──────────────────────────────────────────────────────────
canton_stats = gdf.groupby("canton").apply(
    lambda x: pd.Series({
        "Mit Stockwerk":  x["gastw"].notna().sum(),
        "Ohne Stockwerk": x["gastw"].isna().sum(),
        "Total":          len(x),
    })
).reset_index()

canton_stats["% Mit Stockwerk"] = (
    canton_stats["Mit Stockwerk"] / canton_stats["Total"] * 100
).round(1)
canton_stats = canton_stats.sort_values("% Mit Stockwerk", ascending=False)

print(canton_stats.to_string(index=False))

# ── SINGLE BAR CHART: PERCENT COVERAGE ────────────────────────────────────────
fig = go.Figure(go.Bar(
    name="% Mit Stockwerk",
    x=canton_stats["canton"],
    y=canton_stats["% Mit Stockwerk"],
    text=canton_stats["% Mit Stockwerk"].apply(lambda v: f"{v:.1f}%"),
    textposition="outside",
))

fig.update_traces(cliponaxis=False)
fig.update_layout(
    title={
        "text": "Stockwerk-Abdeckung pro Kanton (% mit Wert)"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                "Quelle: AV/GWR</span>"
    },
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5)
)
fig.update_xaxes(title_text="Kanton")
fig.update_yaxes(title_text="Abdeckung [%]", range=[0, 100])

fig.write_image("stockwerk_coverage_kanton_pct.png")
with open("stockwerk_coverage_kanton_pct.png.meta.json", "w") as f:
    json.dump({
        "caption": "Stockwerk-Abdeckung pro Kanton in Prozent",
        "description": "Bar chart showing percentage of buildings with non-zero gastw per canton"
    }, f)

canton_stats.to_csv("stockwerk_coverage_kanton_pct.csv", index=False)
print("\nSaved → stockwerk_coverage_kanton_pct.png + stockwerk_coverage_kanton_pct.csv")

['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
canton  Mit Stockwerk  Ohne Stockwerk  Total  % Mit Stockwerk
    BS          30463            1784  32247             94.5
    TI         183270           13565 196835             93.1
    BL          97561           31833 129394             75.4
    GE          62044           25477  87521             70.9
    ZH         263748          135368 399116             66.1
    SO          757

hat Volumen und Fläche

In [2]:
import geopandas as gpd
import plotly.graph_objects as go
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file(
    "whatamidoing.geojson"
)

print(gdf.columns.tolist())

# ── COUNT ─────────────────────────────────────────────────────────────────────
total = len(gdf)
has_val = gdf["garea"].notna() & gdf["gvol"].notna()
count_with    = has_val.sum()
count_without = (~has_val).sum()
pct_with    = count_with / total * 100
pct_without = count_without / total * 100

print(f"Total:          {total:,}")
print(f"Mit Wert:       {count_with:,}  ({pct_with:.1f}%)")
print(f"Keine Angabe:   {count_without:,} ({pct_without:.1f}%)")

# ── PIE CHART ─────────────────────────────────────────────────────────────────
fig = go.Figure(go.Pie(
    labels=["Mit Wert", "Keine Angabe"],
    values=[count_with, count_without],
    textinfo="label+percent",
    texttemplate="%{label}<br>%{percent}<br>%{value:,}",
    hole=0.3,
))

fig.update_layout(
    title={
        "text": "Fläche und Volumen – Datenvollständigkeit CH"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                f"Quelle: AV/GWR | Total: {total:,} Gebäude</span>"
    },
    uniformtext_minsize=14,
    uniformtext_mode="hide",
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="right", x=1.1)
)

fig.write_image("vol_coverage.png")
with open("vol_coverage.png.meta.json", "w") as f:
    json.dump({
        "caption": "Datenvollständigkeit Fläche und Volumen",
        "description": "Pie chart showing share of projected buildings with and without area and volume values"
    }, f)


['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
Total:          3,349,331
Mit Wert:       276,507  (8.3%)
Keine Angabe:   3,072,824 (91.7%)


Volumen und Flächen pro Kanton

In [1]:
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file("whatamidoing.geojson")
print(gdf.columns.tolist())

gdf["garea"] = pd.to_numeric(gdf["garea"], errors="coerce")
gdf["gvol"]  = pd.to_numeric(gdf["gvol"],  errors="coerce")

# ── COUNT PER CANTON ──────────────────────────────────────────────────────────
canton_stats = gdf.groupby("canton").apply(
    lambda x: pd.Series({
        "Mit Fläche und Volumen":  (x["garea"].notna() & x["gvol"].notna()).sum(),
        "Ohne Fläche und Volumen": (x["garea"].isna()  | x["gvol"].isna()).sum(),
        "Total":                   len(x),
    })
).reset_index()

canton_stats["% Mit Fläche und Volumen"] = (
    canton_stats["Mit Fläche und Volumen"] / canton_stats["Total"] * 100
).round(1)

# Sort by percentage coverage
canton_stats = canton_stats.sort_values("% Mit Fläche und Volumen", ascending=False)

print(canton_stats.to_string(index=False))

# ── BAR CHART: PERCENT COVERAGE ───────────────────────────────────────────────
fig = go.Figure(go.Bar(
    name="% Mit Fläche und Volumen",
    x=canton_stats["canton"],
    y=canton_stats["% Mit Fläche und Volumen"],
    text=canton_stats["% Mit Fläche und Volumen"].apply(lambda v: f"{v:.1f}%"),
    textposition="outside",
))

fig.update_traces(cliponaxis=False)
fig.update_layout(
    title={
        "text": "Fläche & Volumen – Abdeckung pro Kanton (%)"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                "Quelle: AV/GWR</span>"
    },
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5)
)
fig.update_xaxes(title_text="Kanton")
fig.update_yaxes(title_text="Abdeckung [%]", range=[0, 100])

fig.write_image("vol_coverage_kanton_pct.png")
with open("vol_coverage_kanton_pct.png.meta.json", "w") as f:
    json.dump({
        "caption": "Fläche & Volumen – Abdeckung pro Kanton in Prozent",
        "description": "Bar chart showing percentage of buildings with both garea and gvol per Swiss canton"
    }, f)

canton_stats.to_csv("vol_coverage_kanton_pct.csv", index=False)
print("\nSaved → vol_coverage_kanton_pct.png + vol_coverage_kanton_pct.csv")

['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
canton  Mit Fläche und Volumen  Ohne Fläche und Volumen  Total  % Mit Fläche und Volumen
    BS                   29626                     2621  32247                      91.9
    NW                    5395                    10988  16383                      32.9
    OW                    5553                    16002  21555                      25.8
    VD                   53423     

Anzahl Geschosse Alle Gebäude

In [1]:
import geopandas as gpd
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import json

# ── LOAD DATA FROM GEOJSON ────────────────────────────────────────────────────
gdf = gpd.read_file(
    "whatamidoing.geojson"
)

print(gdf.columns.tolist())


# ── KEY FIX: fill NaN BEFORE value_counts ────────────────────────────────────
col = "gastw"   # change if the name differs in the GeoJSON
series = gdf[col].fillna("Keine Angabe").astype(str)

counts = series.value_counts().reset_index()
counts.columns = ["Geschosse", "Anzahl"]

def sort_key(val):
    try:
        return (0, float(val))
    except:
        return (1, val)

counts = counts.sort_values("Geschosse", key=lambda x: x.map(sort_key)).reset_index(drop=True)

# ── CHART ─────────────────────────────────────────────────────────────────────
fig = go.Figure(go.Bar(
    x=counts["Geschosse"],
    y=counts["Anzahl"],
    textposition="outside",
))

fig.update_traces(cliponaxis=False)
fig.update_layout(
    title={
        "text": "Anzahl Geschosse – Verteilung CH (GeoJSON)"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                "Quelle: projected_buildings_enriched_ch.geojson | inkl. fehlende Werte</span>"
    }
)
fig.update_xaxes(title_text="Geschosse", type="category")
fig.update_yaxes(title_text="Anzahl Gebäude")

fig.write_image("anzahl_geschosse_alle_geojson.png")
with open("anzahl_geschosse_alle_geojson.png.meta.json", "w") as f:
    json.dump({
        "caption": "Verteilung Anzahl Geschosse (GeoJSON, inkl. fehlende Werte)",
        "description": "Bar chart of floor count distribution from projected_buildings_enriched_ch.geojson including missing values"
    }, f)

print(counts.to_string(index=False))

: 

In [15]:
# Load the buildings.geojson file
buildings_gdf = gpd.read_file("../buildings.geojson")

# Display the first 25 entries
display(buildings_gdf.head(25))

,egid,buildingStatus,buildingCategory,buildingClass,municipalityNumber,municipalityName,canton,energyReferenceArea,heating1_heatGenerator,heating1_energySource,...,heating2_revisionDate,hotwater1_heatGenerator,hotwater1_energySource,hotwater1_informationSource,hotwater1_revisionDate,hotwater2_heatGenerator,hotwater2_energySource,hotwater2_informationSource,hotwater2_revisionDate,geometry
0,1,1004,1040.0,1271.0,2,Affoltern am Albis,ZH,NaN,7410.0,7598.0,...,2001-11-29,7630.0,7530.0,860.0,2001-11-29,7600.0,7500.0,860.0,2001-11-29,POINT (2676523 1235843)
1,2,1007,1020.0,1110.0,2,Affoltern am Albis,ZH,NaN,7436.0,7530.0,...,NaT,7630.0,7530.0,860.0,2001-11-29,NaN,NaN,NaN,NaT,POINT (2676541.16 1235979.043)
2,3,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,200.0,7410.0,7501.0,...,2026-02-20,7610.0,7501.0,869.0,2026-02-20,7600.0,7500.0,869.0,2026-02-20,POINT (2676524.002 1236012.212)
3,4,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,NaN,7430.0,7530.0,...,2001-11-29,7630.0,7530.0,860.0,2001-11-29,7600.0,7500.0,860.0,2001-11-29,POINT (2676512.712 1235974.383)
4,5,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,NaN,7430.0,7530.0,...,NaT,7630.0,7530.0,860.0,2001-11-29,7600.0,7500.0,860.0,2001-11-29,POINT (2676502.104 1236014.389)
5,6,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,NaN,7430.0,7530.0,...,2001-11-29,7630.0,7530.0,860.0,2001-11-29,7600.0,7500.0,860.0,2001-11-29,POINT (2676487.05 1235974.052)
6,7,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,160.0,7410.0,7501.0,...,2026-02-20,7610.0,7501.0,869.0,2026-02-20,7600.0,7500.0,869.0,2026-02-20,POINT (2676483.039 1236014.469)
7,8,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,NaN,7410.0,7598.0,...,NaT,7610.0,7598.0,869.0,2022-06-03,NaN,NaN,NaN,NaT,POINT (2676461.803 1235973.58)
8,9,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,NaN,7410.0,7501.0,...,2025-06-06,7610.0,7501.0,869.0,2025-06-06,7600.0,7500.0,869.0,2025-06-06,POINT (2676463.628 1236014.106)
9,10,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,NaN,7410.0,7598.0,...,NaT,7610.0,7598.0,869.0,2018-04-10,NaN,NaN,NaN,NaT,POINT (2676440.038 1235973.538)


In [21]:
import pandas as pd

# Path relative to the notebook's directory
csv_path = "ch/gebaeude_batiment_edificio.csv"

# We only load the columns we need to save memory
cols_to_load = ['GAREA', 'GVOL', 'GKAT', 'GASTW', 'GBAUJ']
df_gwr = pd.read_csv(csv_path, sep='\t', usecols=cols_to_load, low_memory=False)

# Convert GBAUJ to numeric just in case there are weird values
df_gwr['GBAUJ'] = pd.to_numeric(df_gwr['GBAUJ'], errors='coerce')

# Define the three conditions
conditions = [
    ("Built before 2018", df_gwr[df_gwr['GBAUJ'] < 2018]),
    ("Built 2018 to 2021", df_gwr[(df_gwr['GBAUJ'] >= 2018) & (df_gwr['GBAUJ'] <= 2021)]),
    ("Built 2022 to Today", df_gwr[df_gwr['GBAUJ'] >= 2022])
]

columns_to_check = ['GAREA', 'GVOL', 'GKAT', 'GASTW']

for label, subset in conditions:
    print(f"=== {label} ===")
    total = len(subset)
    if total == 0:
        print("No entries found.\n")
        continue
        
    for col in columns_to_check:
        valid_count = subset[col].notna().sum()
        percent = (valid_count / total) * 100
        print(f"{valid_count:,} of {total:,} have a value in {col} which is {percent:.2f}%")
    print() # empty line for better readability


=== Built before 2018 ===
1,795,320 of 1,816,787 have a value in GAREA which is 98.82%
170,406 of 1,816,787 have a value in GVOL which is 9.38%
1,816,555 of 1,816,787 have a value in GKAT which is 99.99%
1,121,332 of 1,816,787 have a value in GASTW which is 61.72%

=== Built 2018 to 2021 ===
74,746 of 75,180 have a value in GAREA which is 99.42%
19,711 of 75,180 have a value in GVOL which is 26.22%
75,180 of 75,180 have a value in GKAT which is 100.00%
59,925 of 75,180 have a value in GASTW which is 79.71%

=== Built 2022 to Today ===
69,423 of 70,298 have a value in GAREA which is 98.76%
52,083 of 70,298 have a value in GVOL which is 74.09%
70,298 of 70,298 have a value in GKAT which is 100.00%
63,231 of 70,298 have a value in GASTW which is 89.95%



In [18]:
import geopandas as gpd

# Load the projected buildings GeoJSON
geojson_proj_path = "../../../Geobasiszwilling/geobasiszwilling_fhnw/import/proj/projected_buildings_enriched_ch.geojson"
print(f"Loading {geojson_proj_path}...")
gdf_proj = gpd.read_file(geojson_proj_path)

# Columns to check for data completeness (assuming lowercase in this geojson)
columns_to_check = ['garea', 'gvol', 'gkat', 'gastw']

print("=== All Projected Buildings (Enriched) ===")
total_proj = len(gdf_proj)

if total_proj == 0:
    print("No entries found.")
else:
    for col in columns_to_check:
        if col in gdf_proj.columns:
            # Mask out None, NaN and empty strings
            valid_mask = gdf_proj[col].notna() & (gdf_proj[col].astype(str).str.strip() != "") & (gdf_proj[col].astype(str).str.strip() != "None")
            valid_count = valid_mask.sum()
            percent = (valid_count / total_proj) * 100
            print(f"{valid_count:,} of {total_proj:,} have a value in {col} which is {percent:.2f}%")
        else:
            print(f"Column '{col}' not found in the GeoJSON.")

Loading ../../../Geobasiszwilling/geobasiszwilling_fhnw/import/proj/projected_buildings_enriched_ch.geojson...
=== All Projected Buildings (Enriched) ===
28,938 of 46,322 have a value in garea which is 62.47%
20,865 of 46,322 have a value in gvol which is 45.04%
29,189 of 46,322 have a value in gkat which is 63.01%
27,613 of 46,322 have a value in gastw which is 59.61%
